In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
import numpy as np
import pandas as pd
import re
from pathlib import Path
from sklearn.model_selection import GroupKFold

In [4]:
# inline stem normalization
_PREFIXES = [
    "Pick the best possible answer:",
    "Select the most accurate option:",
    "Determine the correct option:",
    "Identify the correct statement:",
    "Choose the correct answer:",
    "Which of the following is correct?",
]
_SUFFIXES = [
    "among the listed options.",
    "based on the given context.",
    "from the following choices.",
    "carefully.",
]
_PREFIX_RE = re.compile(r"^\s*(?:" + "|".join(re.escape(p) for p in _PREFIXES) + r")\s*")
_SUFFIX_RE = re.compile(r"\s*(?:" + "|".join(re.escape(s) for s in _SUFFIXES) + r")\s*$")
_WS_RE = re.compile(r"\s+")

def normalize_stem(text):
    if not isinstance(text, str):
        return text
    s = text
    while True:
        before = s
        s = _PREFIX_RE.sub("", s)
        s = _SUFFIX_RE.sub("", s)
        if s == before:
            break
    s = _WS_RE.sub(" ", s).strip()
    return s

In [5]:
# load train, normalize
comp_dir = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
train = pd.read_csv(comp_dir / "train.csv")
train["stem"] = train["prompt"].apply(normalize_stem)
OPT_COLS = ["A", "B", "C", "D", "E"]
L2I = {c: i for i, c in enumerate(OPT_COLS)}
y = train["answer"].map(L2I).values
print(len(train), "rows", train["stem"].nunique(), "unique stems")

2000 rows 252 unique stems


In [6]:
# map3 helper
def map3(scores, labels):
    order = np.argsort(-scores, axis=1)[:, :3]
    ap = np.zeros(len(labels))
    for k in range(3):
        ap += (order[:, k] == labels) / (k + 1)
    return ap.mean()

In [7]:
# answer-length artifact
opt_lens = np.array([[len(str(train.iloc[r][c])) for c in OPT_COLS] for r in range(len(train))], dtype=float)
mask_correct = np.zeros((len(train), 5), bool)
mask_correct[np.arange(len(train)), y] = True
print("mean correct len:", round(opt_lens[mask_correct].mean(), 1))
print("mean incorrect len:", round(opt_lens[~mask_correct].mean(), 1))
print("frac correct == longest:", round((opt_lens.argmax(1) == y).mean(), 4))

mean correct len: 181.9
mean incorrect len: 160.7
frac correct == longest: 0.4055


In [8]:
# trivial baselines under groupkfold
gkf = GroupKFold(n_splits=5)
rng = np.random.RandomState(0)
res = {"random": [], "longest_option": [], "option_text_lookup": []}
for tr, va in gkf.split(train, groups=train["stem"]):
    res["random"].append(map3(rng.rand(len(va), 5), y[va]))
    res["longest_option"].append(map3(opt_lens[va], y[va]))
    m = {}
    for i in tr:
        for c in OPT_COLS:
            t = str(train.iloc[i][c]).strip()
            m.setdefault(t, [0, 0])
            m[t][0] += int(train.iloc[i]["answer"] == c)
            m[t][1] += 1
    prob = {t: a / b for t, (a, b) in m.items()}
    sc = np.full((len(va), 5), -1.0)
    for j, i in enumerate(va):
        for ci, c in enumerate(OPT_COLS):
            sc[j, ci] = prob.get(str(train.iloc[i][c]).strip(), -1.0)
    sc = sc + 1e-6 * opt_lens[va]
    res["option_text_lookup"].append(map3(sc, y[va]))
for k, v in res.items():
    print(f"{k:20s} cv mean MAP@3={np.mean(v):.4f} std={np.std(v):.4f}")

random               cv mean MAP@3=0.3708 std=0.0265
longest_option       cv mean MAP@3=0.5775 std=0.0498
option_text_lookup   cv mean MAP@3=0.6068 std=0.0513
